# Step 03: Category-Stratified Train/Val/Test Split
### Iterative Multi-Label Stratification into 80% Train, 10% Val, and 10% Test
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 2, 3, 6, 9, 10, 11, 12, 13).

### Architectural Guarantees:
- **Equal Proportions**: Preserves exact class distributions across Train, Val, and Test splits.
- **Zero-Copy Architecture**: Images remain in a single shared repository (`coco_files/images/`); split folders contain only lightweight COCO JSON files and soft symlinks.
- **Step B Verification**: Validates 0-indexed contiguous category IDs, split coverage, and box boundaries.

In [ ]:
# STEP 0 - Environment Dependencies
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install --no-cache-dir opencv-python-headless==4.9.0.80
%pip install python-dotenv

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# ==============================================================================
# CELL 2 - MASTER CONFIGURATION & HYPERPARAMETERS
# All pipeline, model, training, evaluation, and inference settings are defined here.
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Dataset & Pipeline Modes
# ------------------------------------------------------------------------------
PIPELINE_NAME = "multi_class_train_rfdetr"
# Full dataset training by default (SAMPLE_SIZE = None). Set an integer (e.g. 1000) for fast testing.
SAMPLE_SIZE = None
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"
RANDOM_SEED = 42

# ------------------------------------------------------------------------------
# Model Versioning & Experiment Isolation
# Allows training new model versions without touching or overwriting old models/checkpoints
# ------------------------------------------------------------------------------
MODEL_VERSION = "v2"      # Set version tag: "v2", "v3", etc. (Leave empty "" for legacy baseline)
RUN_ID = f"{MODE_TAG}_{MODEL_VERSION}" if MODEL_VERSION else MODE_TAG

# Dataset Split Proportions
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO  = 0.10

# COCO Annotation Field Names
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# Azure Storage Download Settings
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"
DOWNLOAD_WORKERS = 16

# ------------------------------------------------------------------------------
# ------------------------------------------------------------------------------
# ------------------------------------------------------------------------------
# 2. Model Architecture & Automatic Resolution Calculation
# ------------------------------------------------------------------------------
MODEL_SIZE = "base"      # Options: 'nano', 'small', 'medium', 'base', 'large'
FREEZE_BACKBONE = True   # Freeze DINOv2 ViT backbone: cuts 1 epoch from ~2h to ~30-35m (RECOMMENDED)
PRETRAINED = True

def get_backbone_divisor(model_size: str) -> int:
    """
    Automatically determines the required spatial divisibility (block_size = patch_size * num_windows):
      - 'base': patch_size 14 * num_windows 4 = 56
      - 'nano', 'small', 'medium', 'large': patch_size 16 * num_windows 2 = 32
    """
    m = str(model_size).lower().strip()
    if m == "base":
        return 56
    elif m in ["nano", "small", "medium", "large"]:
        return 32
    return 56

def calculate_valid_resolution(target: int, model_size: str = MODEL_SIZE, universal: bool = False) -> int:
    """
    Automatically calculates the nearest valid resolution for the model backbone.
    - If universal=True: uses LCM(32, 56) = 224 (e.g. 896, 1120), guaranteed compatible with ALL models.
    - Otherwise: uses the exact divisor required by MODEL_SIZE (56 for 'base', 32 for others).
    """
    divisor = 224 if universal else get_backbone_divisor(model_size)
    valid = round(target / divisor) * divisor
    return max(valid, divisor)

# Desired target resolution (e.g. 1008, 1000, 896)
TARGET_RESOLUTION = 1008
BACKBONE_DIVISOR = get_backbone_divisor(MODEL_SIZE)  # 56 for base, 32 for small/medium/nano/large

# Automatic calculation: Computes resolution matching the chosen model architecture
RESOLUTION = calculate_valid_resolution(TARGET_RESOLUTION, MODEL_SIZE)
PRETRAINED_WEIGHTS = READABLE_WEIGHTS_PATH if READABLE_WEIGHTS_PATH else "rf-detr-base-coco.pth"
# 3. Training & Throughput Optimization (Optimized for Fast Epochs on 40k images)
# ------------------------------------------------------------------------------
EPOCHS = 10              # User constraint: max 10 epochs
# Resume Control: When starting a new model version (v2), default to starting fresh from pretrained backbone
# Set RESUME_TRAINING = True ONLY if you want to resume an interrupted v2 run from latest_checkpoint.pth
RESUME_TRAINING = False         # False = start fresh v2 training; True = resume v2 checkpoint
RESUME_CHECKPOINT_PATH = None   # None = auto-detect latest_checkpoint.pth in v2 CHECKPOINT_DIR
BATCH_SIZE = 4           # Training batch size (fits safely in 16GB T4 VRAM with frozen backbone)
GRAD_ACCUMULATION = 4    # Gradient accumulation steps
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUMULATION  # Automatic calculation: Effective batch = 16
STEPS_PER_EPOCH = 2000   # 2,000 steps per epoch (~15 mins/epoch, 20k steps total over 10 epochs)
VAL_BATCH_SIZE = 2       # User constraint: Validation batch size = 2
NUM_WORKERS = 2         # User constraint: max 2 workers
PIN_MEMORY = True        # Pin CUDA memory for zero-copy host-to-device transfers
PERSISTENT_WORKERS = True# Keep DataLoader workers alive between epochs to avoid respawn latency
PREFETCH_FACTOR = 2      # Batches prefetched per worker

# Optimizer & Learning Rate Schedule
LEARNING_RATE = 1e-4     # Base learning rate for AdamW
WEIGHT_DECAY = 1e-4      # L2 regularization
LR_MIN = 1e-6            # Minimum LR for CosineAnnealing schedule
CLIP_GRAD_MAX_NORM = 0.1 # Maximum gradient norm clipping

# ------------------------------------------------------------------------------
# 4. Regularization, Augmentation & Early Stopping / Overfitting Controls
# ------------------------------------------------------------------------------
USE_AUGMENTATION = True        # Step C: Enable Random Horizontal Flip & Color Jitter for train split
EARLY_STOPPING_PATIENCE = 5    # Stop if Val Loss does not improve for 5 epochs
STOP_ON_OVERFITTING = True      # Automatically halt training when overfitting is detected
OVERFITTING_RATIO_ALERT = 2.0  # Log warning notice when Val_Loss / Train_Loss exceeds 2.0x
OVERFITTING_RATIO_THRESHOLD = 2.5 # Val_Loss / Train_Loss ratio threshold to count towards overfitting stop
OVERFITTING_PATIENCE = 2        # Stop after 2 consecutive overfitted epochs
OVERFITTING_WARMUP_EPOCHS = 3   # Grace period: ignore overfitting stops during first 3 warmup epochs
EVAL_MAP_INTERVAL = 1          # Evaluates mAP@50, mAP@75, mAP@50:95, and mAR on every epoch
MAP_EVAL_THRESHOLD = 0.01      # Confidence threshold for mAP (0.01 captures full Precision-Recall curve)

# ------------------------------------------------------------------------------
# 5. Inference, Post-Processing & Visualization
# ------------------------------------------------------------------------------
CONFIDENCE = 0.50              # Minimum confidence score for test detections
CONFIDENCE_THRESHOLD = CONFIDENCE
NMS_THRESHOLD = 0.50           # NMS IoU threshold
NUM_VISUAL_SAMPLES = 4         # Number of ground-truth samples to plot in Cell 13
NUM_INFERENCE_PREVIEWS = 3     # Number of test detection visual plots in Cell 21

# ------------------------------------------------------------------------------
# 6. Directory Structure & File Paths
# ------------------------------------------------------------------------------
REPO_ROOT = Path(".")
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

DATASET_DIR = os.path.join(PIPELINE_DIR, f"dataset_{MODE_TAG}")
IMAGES_DIR = PIPELINE_DIR / "images"
RAW_IMAGES_DIR = IMAGES_DIR
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(DATASET_DIR, s) for s in ["train", "val", "test"]]
TRAIN_ANN, VAL_ANN, TEST_ANN = [os.path.join(d, "_annotations.coco.json") for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]]

OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"runs/rf_detr_{RUN_ID}")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
INFERENCE_OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"inference_{RUN_ID}")
FINAL_MODEL_DIR = os.path.join(PIPELINE_DIR, "model")

# Version-specific export model paths (guarantees old models remain 100% untouched)
EXPORT_MODEL_NAME = f"best_model_{RUN_ID}.pth"
EXPORTED_MODEL_PATH = os.path.join(FINAL_MODEL_DIR, EXPORT_MODEL_NAME)

# ------------------------------------------------------------------------------
# 7. Hardware & Kernel Acceleration Settings
# ------------------------------------------------------------------------------
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")  # Fast TensorFloat32 math on Ampere/Ada/Hopper

logger.info("=" * 75)
logger.info(f"Master Configuration Loaded: Pipeline={PIPELINE_NAME} ({MODE_TAG}) | Version={MODEL_VERSION} (RUN_ID={RUN_ID})")
logger.info(f"   [ISOLATION] Output Directory: {OUTPUT_DIR} (Old model checkpoints are 100% PRESERVED)")
logger.info(f"   - Mode:          {'Full Dataset (40k+)' if SAMPLE_SIZE is None else f'Sample Mode ({SAMPLE_SIZE} images)'}")
logger.info(f"   - Batch Size:    Train={BATCH_SIZE} (Eff={BATCH_SIZE * GRAD_ACCUMULATION}), Val={VAL_BATCH_SIZE}, Steps/Epoch={STEPS_PER_EPOCH if STEPS_PER_EPOCH else 'Full'}")
logger.info(f"   - Throughput:    Workers={NUM_WORKERS}, Resolution={RESOLUTION}x{RESOLUTION}, PinMem={PIN_MEMORY}")
logger.info(f"   - Optimization:  LR={LEARNING_RATE}, WeightDecay={WEIGHT_DECAY}, ClipNorm={CLIP_GRAD_MAX_NORM}")
logger.info(f"   - Diagnostics:   FreezeBackbone={FREEZE_BACKBONE}, StopOnOverfit={STOP_ON_OVERFITTING} (ratio={OVERFITTING_RATIO_THRESHOLD}x, pat={OVERFITTING_PATIENCE}), Aug={USE_AUGMENTATION}, Patience={EARLY_STOPPING_PATIENCE}")

In [ ]:
# CELL 3 - Dynamic Folder Creation & File Logging
for p in [DATASET_DIR, IMAGES_DIR, OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, INFERENCE_OUTPUT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

log_file_path = os.path.join(LOG_DIR, "pipeline.log")
for h in [h for h in logger.handlers if isinstance(h, (logging.FileHandler, FlushFileHandler))]:
    logger.removeHandler(h); h.close()

file_handler = FlushFileHandler(log_file_path, mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

def tail_log(n: int = 25):
    """View latest log entries from inside Jupyter."""
    if os.path.exists(log_file_path):
        with open(log_file_path, "r", encoding="utf-8") as f:
            print("".join(f.readlines()[-n:]))

load_dotenv()
conn_str = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(conn_str) if conn_str else None
except Exception as e:
    blob_service_client = None

logger.info(f"Directories ready. Active file logger writing to: {log_file_path}")

In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 6 - Group Annotations & Apply Stratified Sampling Mode
logger.info("=" * 70)
logger.info(f"[CELL 6] Applying mode selection: {MODE_TAG.upper()}")
logger.info("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

image_primary_cat: Dict[str, str] = {}
for u in all_image_urls:
    anns = image_annotations[u]
    if anns:
        cats = [a["category_name"] for a in anns]
        image_primary_cat[u] = max(set(cats), key=cats.count)
    else:
        image_primary_cat[u] = AUTO_CATEGORIES[0]

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = []
    cat_to_images: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in all_image_urls:
        cat_to_images[image_primary_cat[u]].append(u)
    
    total_imgs = len(all_image_urls)
    for cat in AUTO_CATEGORIES:
        imgs = cat_to_images[cat]
        target_count = max(1, int(round((len(imgs) / total_imgs) * SAMPLE_SIZE)))
        selected = random.sample(imgs, min(len(imgs), target_count))
        selected_image_urls.extend(selected)
    
    if len(selected_image_urls) > SAMPLE_SIZE:
        selected_image_urls = random.sample(selected_image_urls, SAMPLE_SIZE)
    logger.info(f"   Sample Mode Active: Stratified selection of {len(selected_image_urls)} images.")
else:
    selected_image_urls = all_image_urls
    logger.info(f"   Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(list(set(selected_image_urls)))
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

logger.info(f"   - Total images in pipeline:         {len(image_urls)}")
logger.info(f"   - Total bounding boxes in pipeline: {total_selected_boxes}")

In [ ]:
# CELL 9 - Category-Stratified Train/Val/Test Split (Equal Proportions)
logger.info("=" * 70)
logger.info("[CELL 9] Performing Category-Stratified Split (Train 80%, Val 10%, Test 10%)...")
logger.info("=" * 70)

# Check if existing split annotations already exist on disk to guarantee 100% consistency
split_files_exist = all(os.path.exists(p) and os.path.getsize(p) > 100 for p in [TRAIN_ANN, VAL_ANN, TEST_ANN])

if split_files_exist:
    logger.info("=" * 70)
    logger.info(f"[DATASET CONSISTENCY] Existing Train/Val/Test splits found in {DATASET_DIR}!")
    logger.info("Reusing EXACT existing splits: Train, Val, and Test remain 100% identical and unchanged.")
    logger.info("=" * 70)
    
    with open(TRAIN_ANN, "r", encoding="utf-8") as f:
        train_coco = json.load(f)
    with open(VAL_ANN, "r", encoding="utf-8") as f:
        val_coco = json.load(f)
    with open(TEST_ANN, "r", encoding="utf-8") as f:
        test_coco = json.load(f)
        
    train_urls = [im.get("original_url", im["file_name"]) for im in train_coco["images"]]
    val_urls = [im.get("original_url", im["file_name"]) for im in val_coco["images"]]
    test_urls = [im.get("original_url", im["file_name"]) for im in test_coco["images"]]
    
    logger.info(f"   - Train split: {len(train_urls):,} images (100% preserved from existing split)")
    logger.info(f"   - Val split:   {len(val_urls):,} images (100% preserved from existing split)")
    logger.info(f"   - Test split:  {len(test_urls):,} images (100% preserved from existing split)")
else:
    logger.info("[CELL 9] Creating initial Category-Stratified Split using fixed RANDOM_SEED (42)...")
    valid_urls = [u for u in image_urls if download_results.get(u, {}).get("error") is None]
    random.seed(RANDOM_SEED)

    cat_to_valid_imgs: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in valid_urls:
        p_cat = image_primary_cat.get(u, AUTO_CATEGORIES[0])
        cat_to_valid_imgs[p_cat].append(u)

    train_urls, val_urls, test_urls = [], [], []
    for cat, imgs in cat_to_valid_imgs.items():
        random.shuffle(imgs)
        n = len(imgs)
        n_tr = int(n * TRAIN_RATIO)
        n_va = int(n * VALID_RATIO)
        train_urls.extend(imgs[:n_tr])
        val_urls.extend(imgs[n_tr:n_tr + n_va])
        test_urls.extend(imgs[n_tr + n_va:])

def count_category_boxes(urls: List[str]) -> Dict[str, int]:
    counts = {cat: 0 for cat in AUTO_CATEGORIES}
    for u in urls:
        for a in filtered_annotations.get(u, []):
            c = a["category_name"]
            counts[c] = counts.get(c, 0) + 1
    return counts

train_cat_boxes = count_category_boxes(train_urls)
val_cat_boxes = count_category_boxes(val_urls)
test_cat_boxes = count_category_boxes(test_urls)
total_cat_boxes = {cat: train_cat_boxes[cat] + val_cat_boxes[cat] + test_cat_boxes[cat] for cat in AUTO_CATEGORIES}

proportions_rows = []
for cat in AUTO_CATEGORIES:
    tot = max(1, total_cat_boxes[cat])
    proportions_rows.append({
        "Category": cat,
        "Total Boxes": total_cat_boxes[cat],
        "Train Count": train_cat_boxes[cat],
        "Train %": f"{(train_cat_boxes[cat]/tot)*100:.1f}%",
        "Val Count": val_cat_boxes[cat],
        "Val %": f"{(val_cat_boxes[cat]/tot)*100:.1f}%",
        "Test Count": test_cat_boxes[cat],
        "Test %": f"{(test_cat_boxes[cat]/tot)*100:.1f}%",
    })

proportions_df = pd.DataFrame(proportions_rows)
logger.info("Category Proportions Across Splits:")
logger.info("\n" + proportions_df.to_string(index=False))
logger.info(f"Total Images: Train={len(train_urls)}, Val={len(val_urls)}, Test={len(test_urls)}")

In [ ]:
# CELL 10 - Prepare Split Annotation Directories (Zero-Copy Architecture)
logger.info("=" * 70)
logger.info("[CELL 10] Preparing split annotation directories (Zero-Copy Architecture)...")
logger.info("=" * 70)

for split_name in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(split_name, exist_ok=True)
    images_link = os.path.join(split_name, "images")
    if not os.path.exists(images_link):
        try:
            os.symlink(os.path.abspath(IMAGES_DIR), images_link)
        except OSError:
            pass

logger.info(f"   - Shared images directory:  {IMAGES_DIR} ({len(os.listdir(IMAGES_DIR))} files)")
logger.info(f"   - Train split annotations:  {TRAIN_DIR} ({len(train_urls)} images)")
logger.info(f"   - Val split annotations:    {VAL_DIR} ({len(val_urls)} images)")
logger.info(f"   - Test split annotations:   {TEST_DIR} ({len(test_urls)} images)")
logger.info("Split directories ready with zero redundant image copying.")

In [ ]:
# CELL 11 - Build Multi-Class COCO Annotations
logger.info("=" * 70)
logger.info(f"[CELL 11] Building COCO annotations for {NUM_CLASSES} discovered categories...")
logger.info("=" * 70)

categories_def = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]

def build_coco_for_split(urls: List[str], split_dir: str, ann_path: str) -> Dict[str, Any]:
    images_list = []
    annotations_list = []
    ann_id = 1

    for img_id_idx, url in enumerate(urls, start=1):
        local_path = download_results[url]["local_path"]
        if not (local_path and local_path.exists()):
            continue
        try:
            with Image.open(local_path) as im:
                width, height = im.size
        except Exception:
            width, height = 1920, 1080

        filename = local_path.name
        images_list.append({
            "id": img_id_idx,
            "file_name": filename,
            "width": int(width),
            "height": int(height),
            "original_url": url
        })

        for ann in filtered_annotations.get(url, []):
            x, y, w, h = ann["bbox"]
            cid = ann["category_id"]
            annotations_list.append({
                "id": ann_id,
                "image_id": img_id_idx,
                "category_id": int(cid),
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            ann_id += 1

    coco_dict = {
        "info": {"description": f"RF-DETR Multi-Class Dataset ({NUM_CLASSES} classes)", "version": "1.0"},
        "licenses": [],
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories_def
    }
    
    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f, indent=2)
    logger.info(f"   - Saved: {ann_path} ({len(images_list)} images, {len(annotations_list)} boxes)")
    return coco_dict

# Only build if split files do not already exist (preserves existing dataset splits 100% intact)
split_files_exist = all(os.path.exists(p) and os.path.getsize(p) > 100 for p in [TRAIN_ANN, VAL_ANN, TEST_ANN])

if split_files_exist:
    logger.info("=" * 70)
    logger.info("[DATASET INTEGRITY] Train, Val, and Test split annotations already exist.")
    logger.info("Preserving existing split files 100% untouched to ensure exact consistency across model versions.")
    logger.info(f"   - Train: {TRAIN_ANN}")
    logger.info(f"   - Val:   {VAL_ANN}")
    logger.info(f"   - Test:  {TEST_ANN}")
    logger.info("=" * 70)
else:
    build_coco_for_split(train_urls, TRAIN_DIR, TRAIN_ANN)
    build_coco_for_split(val_urls, VAL_DIR, VAL_ANN)
    build_coco_for_split(test_urls, TEST_DIR, TEST_ANN)

In [ ]:
# CELL 12 - Dataset Inspection & Step B Label/Category Alignment Verification
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
logger.info("All annotation files verified.")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

logger.info("=" * 58)
logger.info(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
logger.info("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    logger.info(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
logger.info("=" * 58)
logger.info(f"Classes ({NUM_CLASSES} total): {train_info['categories']}")

# ==========================================================================
# STEP B CHECKS: Comprehensive Label & Category Alignment Diagnostic Suite
# ==========================================================================
from collections import defaultdict, Counter
def verify_label_and_category_alignment(train_path: str, val_path: str, test_path: str) -> dict:
    """Rigorous Step B diagnostic check verifying category indexing, split coverage, and box coordinates."""
    logger.info("=" * 75)
    logger.info("[STEP B CHECK] Executing Label & Category Alignment Verification Suite...")
    logger.info("=" * 75)
    
    issues = []
    splits = {"train": train_path, "val": val_path, "test": test_path}
    loaded = {}
    for sname, spath in splits.items():
        with open(spath, "r", encoding="utf-8") as f:
            loaded[sname] = json.load(f)

    # Check 1: Contiguous 0-indexing in categories
    train_cats = loaded["train"].get("categories", [])
    train_ids = sorted([c["id"] for c in train_cats])
    expected_ids = list(range(len(train_cats)))
    if train_ids != expected_ids:
        msg = f"Category IDs are not contiguous 0-indexed! Expected {expected_ids[:5]}..., got {train_ids[:5]}..."
        issues.append(msg)
        logger.error(f"   [FAIL] {msg}")
    else:
        logger.info(f"   [PASS] Contiguous 0-indexing verified: IDs {min(train_ids)} to {max(train_ids)} ({len(train_ids)} classes).")

    # Check 2: Cross-split category consistency
    train_cat_dict = {c["id"]: c["name"] for c in train_cats}
    for sname in ["val", "test"]:
        s_ids = set(c["id"] for c in loaded[sname].get("categories", []))
        diff = s_ids - set(train_ids)
        if diff:
            msg = f"{sname.capitalize()} contains category IDs not present in Train: {diff}"
            issues.append(msg)
            logger.error(f"   [FAIL] {msg}")
        else:
            logger.info(f"   [PASS] All {sname} categories align exactly with train categories.")

    # Check 3: Bounding box validity and class counts
    ann_counts = {s: defaultdict(int) for s in splits}
    degenerate_boxes = 0
    for sname, data in loaded.items():
        imgs = {im["id"]: im for im in data.get("images", [])}
        for ann in data.get("annotations", []):
            cid = ann.get("category_id")
            ann_counts[sname][cid] += 1
            bbox = ann.get("bbox", [])
            if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
                degenerate_boxes += 1
                continue
            x, y, w, h = bbox
            im_meta = imgs.get(ann.get("image_id"), {})
            im_w, im_h = im_meta.get("width", 1920), im_meta.get("height", 1080)
            if x < 0 or y < 0 or (x + w) > im_w * 1.05 or (y + h) > im_h * 1.05:
                degenerate_boxes += 1

    logger.info("-" * 75)
    logger.info(f"{'ID':<5} {'Category Name':<25} {'Train Boxes':>12} {'Val Boxes':>11} {'Test Boxes':>11} {'Status':>8}")
    logger.info("-" * 75)
    for cid in train_ids:
        cname = train_cat_dict.get(cid, "Unknown")
        tr_c = ann_counts["train"][cid]
        va_c = ann_counts["val"][cid]
        te_c = ann_counts["test"][cid]
        status = "OK"
        if va_c == 0:
            status = "NO VAL!"
            issues.append(f"Category '{cname}' (ID {cid}) has 0 instances in validation split!")
        elif tr_c < 5:
            status = "LOW DATA"
            issues.append(f"Category '{cname}' (ID {cid}) has only {tr_c} train instances (<5).")
        logger.info(f"{cid:<5} {cname[:24]:<25} {tr_c:>12} {va_c:>11} {te_c:>11} {status:>8}")
    logger.info("-" * 75)

    if degenerate_boxes > 0:
        msg = f"Detected {degenerate_boxes} degenerate or out-of-bounds bounding boxes across dataset splits!"
        issues.append(msg)
        logger.warning(f"   [WARN] {msg}")
    else:
        logger.info("   [PASS] All bounding boxes verified: valid positive width/height within image limits.")

    if not issues:
        logger.info("[STEP B CHECK PASSED] Dataset labels, indexing, and splits are fully consistent.")
    else:
        logger.warning(f"[STEP B CHECK NOTICE] {len(issues)} item(s) to be aware of. Validation loss may be affected by class distribution.")
    logger.info("=" * 75)
    return {"issues": issues, "ann_counts": ann_counts}

step_b_audit = verify_label_and_category_alignment(TRAIN_ANN, VAL_ANN, TEST_ANN)

In [ ]:
# CELL 13 - Print Sample Images per Tag (In-Memory Display - No Files Saved)
from PIL import Image, ImageDraw
from IPython.display import display
from collections import defaultdict

# High-contrast color palette for distinct multi-class boxes
SAMPLE_COLORS = [
    (230, 25, 75),   # Red
    (60, 180, 75),   # Green
    (255, 225, 25),  # Yellow
    (67, 99, 216),   # Blue
    (245, 130, 49),  # Orange
    (145, 30, 180),  # Purple
    (66, 212, 244),  # Cyan
    (240, 50, 230),  # Magenta
    (191, 239, 69),  # Lime
    (250, 190, 212), # Pink
    (70, 153, 144),  # Teal
    (220, 190, 255)  # Lavender
]

def print_sample_images_per_tag(annotation_path: str, image_dir: str, max_display_width: int = 700):
    """
    Selects sample images covering each discovered category tag, annotates
    bounding boxes with colored badges in memory, and prints/displays them
    inline directly in the notebook without saving any files to disk.
    """
    print("=" * 75)
    print("[CELL 13] Printing Sample Images with Category Tags (No Disk Saving)")
    print("=" * 75)

    with open(annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    img_id_map = {im["id"]: im for im in data.get("images", [])}
    
    cat_to_images = defaultdict(list)
    img_to_anns = defaultdict(list)
    for ann in data.get("annotations", []):
        cid = ann.get("category_id")
        iid = ann.get("image_id")
        cat_to_images[cid].append(iid)
        img_to_anns[iid].append(ann)

    # Match with existing local image files
    existing_cat_images = {}
    for cid, iids in cat_to_images.items():
        valid_iids = [
            iid for iid in iids
            if iid in img_id_map and os.path.exists(os.path.join(image_dir, img_id_map[iid]["file_name"]))
        ]
        if valid_iids:
            existing_cat_images[cid] = valid_iids

    if not existing_cat_images:
        print("[WARN] No valid local images found to display.")
        return

    # Display an annotated sample for every distinct category tag
    for cid in sorted(existing_cat_images.keys()):
        cat_name = categories.get(cid, str(cid))
        sample_img_id = existing_cat_images[cid][0]
        meta = img_id_map[sample_img_id]
        fname = meta["file_name"]
        img_path = os.path.join(image_dir, fname)

        img = Image.open(img_path).convert("RGB")
        im_w, im_h = img.size
        draw = ImageDraw.Draw(img)
        anns = img_to_anns.get(sample_img_id, [])

        cat_counts = defaultdict(int)
        for ann in anns:
            raw_bbox = ann.get("bbox", [])
            if len(raw_bbox) != 4:
                continue
            x, y, w, h = [float(v) for v in raw_bbox]
            ann_cid = int(ann.get("category_id", 0))
            ann_cname = str(categories.get(ann_cid, ann_cid))
            cat_counts[ann_cname] += 1
            color = SAMPLE_COLORS[ann_cid % len(SAMPLE_COLORS)]

            # Clamp box to valid dimensions
            x0 = max(0.0, min(float(im_w - 1), x))
            y0 = max(0.0, min(float(im_h - 1), y))
            x1 = max(0.0, min(float(im_w - 1), x + w))
            y1 = max(0.0, min(float(im_h - 1), y + h))

            # Draw bounding box
            draw.rectangle([x0, y0, x1, y1], outline=color, width=4)

            # Draw colored label badge with tag name
            label_text = f" {ann_cname} "
            if hasattr(draw, "textbbox"):
                tb = draw.textbbox((x0, max(0.0, y0 - 18.0)), label_text)
                draw.rectangle(tb, fill=color)
                draw.text((x0, max(0.0, y0 - 18.0)), label_text, fill=(255, 255, 255))
            else:
                draw.text((x0 + 2, max(0.0, y0 - 14.0)), label_text, fill=color)

        summary_str = ", ".join(f"'{k}': {v}" for k, v in sorted(cat_counts.items()))
        print(f"\n>>> Target Tag: [{cat_name}] | File: {fname} ({im_w}x{im_h}) | Boxes: {len(anns)} ({summary_str})")

        # Scale in-memory copy for clean notebook view if wide
        if im_w > max_display_width:
            scale = max_display_width / im_w
            display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
        else:
            display_img = img

        # Directly print/display in notebook (Zero files saved to disk)
        display(display_img)

    print("\n" + "=" * 75)
    print("All category tag samples displayed inline successfully.")
    print("=" * 75)

# Aliases for compatibility with earlier calls
visualize_samples = lambda *args, **kwargs: print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))
print_and_visualize_samples = visualize_samples

print_sample_images_per_tag(TRAIN_ANN, str(IMAGES_DIR))